# 02 — Matchup Stats
Analyse détaillée de tes matchups par champion adverse.

In [ ]:
import sys
sys.path.insert(0, '../..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pipeline.load_db import get_games_as_df

sns.set_theme(style='whitegrid')
df = get_games_as_df()
df = df[df['opponent_champion_name'].notna()]
print(f'{len(df)} games with opponent data')

In [ ]:
# Matchup matrix: winrate per (my_champion, opponent)
matchup = df.groupby(['champion_name', 'opponent_champion_name']).agg(
    games=('win', 'count'),
    winrate=('win', 'mean'),
    avg_kda=('kda_ratio', 'mean'),
    avg_cs=('cs_per_min', 'mean'),
).round(3).reset_index()

matchup_filtered = matchup[matchup['games'] >= 2]
matchup_filtered.sort_values(['champion_name', 'winrate'], ascending=[True, False])

In [ ]:
# Heatmap winrate: champion vs opponent (min 2 games)
MY_POOL = ['Orianna', 'Ahri', 'Galio', 'Mel', 'Anivia']
pool_df = df[df['champion_name'].isin(MY_POOL)]

pivot = pool_df.pivot_table(
    values='win',
    index='champion_name',
    columns='opponent_champion_name',
    aggfunc='mean'
)

# Filter columns with at least 2 data points
counts = pool_df.pivot_table(
    values='win',
    index='champion_name',
    columns='opponent_champion_name',
    aggfunc='count'
)
pivot = pivot.where(counts >= 2)

if not pivot.empty and pivot.notna().any().any():
    fig, ax = plt.subplots(figsize=(max(10, len(pivot.columns)), max(4, len(pivot))))
    sns.heatmap(
        pivot, annot=True, fmt='.0%', cmap='RdYlGn',
        center=0.5, vmin=0, vmax=1,
        linewidths=0.5, ax=ax,
        cbar_kws={'label': 'Winrate'}
    )
    ax.set_title('Matchup heatmap — winrate par adversaire (min. 2 games)')
    ax.set_xlabel('Adversaire mid')
    ax.set_ylabel('Mon champion')
    plt.tight_layout()
    plt.show()
else:
    print('Pas encore assez de données pour la heatmap (min 2 games par matchup).')

In [ ]:
# Top difficult matchups (lowest winrate, ≥2 games)
FOCUS_CHAMP = 'Ahri'  # Change this

champ_matchups = matchup_filtered[matchup_filtered['champion_name'] == FOCUS_CHAMP]

if not champ_matchups.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Worst matchups
    worst = champ_matchups.nsmallest(8, 'winrate')
    axes[0].barh(worst['opponent_champion_name'], worst['winrate'],
                 color='#E74C3C')
    axes[0].axvline(0.5, color='gray', linestyle='--')
    axes[0].set_title(f'{FOCUS_CHAMP} — matchups difficiles')
    axes[0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))

    # Best matchups
    best = champ_matchups.nlargest(8, 'winrate')
    axes[1].barh(best['opponent_champion_name'], best['winrate'],
                 color='#27AE60')
    axes[1].axvline(0.5, color='gray', linestyle='--')
    axes[1].set_title(f'{FOCUS_CHAMP} — matchups favorables')
    axes[1].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))

    plt.tight_layout()
    plt.show()
else:
    print(f'Pas assez de données pour {FOCUS_CHAMP}.')

In [ ]:
# CS/min win vs loss comparison per champion
cs_comp = df.groupby(['champion_name', 'win'])['cs_per_min'].mean().unstack()
cs_comp.columns = ['Défaites', 'Victoires']
cs_comp = cs_comp.dropna()

if not cs_comp.empty:
    ax = cs_comp.plot(kind='bar', figsize=(10, 5), color=['#E74C3C', '#27AE60'])
    ax.set_title('CS/min moyen : victoires vs défaites')
    ax.set_ylabel('CS/min')
    ax.set_xlabel('')
    ax.legend()
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()